# EDA — Amazon Computers

Exploratory analysis of the Amazon Co-Buy Computer node classification graph, loaded through this repo's own `common/data.py` (same loader every training script uses) and evaluated against the fixed `split_idx.csv` split, so the numbers here match what the benchmark scripts actually train/evaluate on.

In [ ]:
# Fresh clone that hasn't run `pip install -r requirements.txt` yet? Uncomment:
# %pip install -q -r ../requirements.txt

## 1. Setup

In [ ]:
import json
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.data import load_products, load_split_idx_csv

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger("eda")

OUT_DIR = REPO_ROOT / "outputs" / "eda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load dataset + fixed split

In [ ]:
data, labels, split_idx, num_classes = load_products(REPO_ROOT / "data", logger)

split_file = REPO_ROOT / "split_idx.csv"
if split_file.is_file():
    split_idx = load_split_idx_csv(split_file)
    logger.info("Using fixed split from %s", split_file)

num_nodes = data.num_nodes
num_edges = data.edge_index.size(1)  # directed edge count (graph is symmetrised, so this is 2x the undirected count)
num_features = data.x.size(1)
split_sizes = {k: int(v.numel()) for k, v in split_idx.items()}

summary = {
    "num_nodes": int(num_nodes),
    "num_edges_directed": int(num_edges),
    "num_node_features": int(num_features),
    "num_classes": int(num_classes),
    "split_sizes": split_sizes,
}
print(json.dumps(summary, indent=2))
(OUT_DIR / "dataset_summary.json").write_text(json.dumps(summary, indent=2))

## 3. Split sizes

In [ ]:
split_df = pd.DataFrame([
    {"split": k, "nodes": v, "percent": 100 * v / num_nodes}
    for k, v in split_sizes.items()
])
display(split_df)

ax = split_df.plot(kind="bar", x="split", y="nodes", legend=False, title="Split sizes", figsize=(6, 4))
ax.set_ylabel("nodes")
plt.tight_layout()
plt.savefig(OUT_DIR / "split_sizes.png", dpi=150)
plt.show()

## 4. Label distribution

In [ ]:
label_counts = torch.bincount(labels, minlength=num_classes).numpy()
label_df = pd.DataFrame({"class": np.arange(num_classes), "count": label_counts})
label_df["percent"] = 100 * label_df["count"] / label_df["count"].sum()
display(label_df.sort_values("count", ascending=False))
label_df.to_csv(OUT_DIR / "label_distribution.csv", index=False)

rows = []
for split_name, idx in split_idx.items():
    counts = torch.bincount(labels[idx], minlength=num_classes).numpy()
    for cls, count in enumerate(counts):
        rows.append({"split": split_name, "class": cls, "count": int(count)})
split_label_df = pd.DataFrame(rows)
pivot = split_label_df.pivot(index="class", columns="split", values="count")

pivot.plot(kind="bar", figsize=(12, 4.5), title="Class distribution by split")
plt.xlabel("class")
plt.ylabel("nodes")
plt.tight_layout()
plt.savefig(OUT_DIR / "split_label_distribution.png", dpi=150)
plt.show()

## 5. Feature statistics

Node features are bag-of-words counts over product reviews — expect a sparse, right-skewed distribution.

In [ ]:
x = data.x
feature_stats = {
    "mean": float(x.mean()),
    "std": float(x.std()),
    "min": float(x.min()),
    "max": float(x.max()),
    "zero_fraction": float((x == 0).float().mean()),
}
print(json.dumps(feature_stats, indent=2))
(OUT_DIR / "feature_stats.json").write_text(json.dumps(feature_stats, indent=2))

sample_idx = torch.randperm(num_nodes)[:min(5000, num_nodes)]
plt.figure(figsize=(7, 4))
plt.hist(x[sample_idx].flatten().numpy(), bins=80)
plt.title(f"Feature value histogram (sampled {sample_idx.numel():,} nodes)")
plt.xlabel("feature value")
plt.ylabel("frequency")
plt.yscale("log")
plt.tight_layout()
plt.savefig(OUT_DIR / "feature_value_histogram.png", dpi=150)
plt.show()

## 6. Degree statistics

In [ ]:
row, col = data.edge_index
degree = torch.bincount(row, minlength=num_nodes)

degree_stats = {
    "mean": float(degree.float().mean()),
    "median": float(degree.float().median()),
    "max": int(degree.max()),
    "isolated_nodes": int((degree == 0).sum()),
}
print(json.dumps(degree_stats, indent=2))
(OUT_DIR / "degree_stats.json").write_text(json.dumps(degree_stats, indent=2))

degree_df = pd.DataFrame({"node": np.arange(num_nodes), "degree": degree.numpy(), "label": labels.numpy()})

plt.figure(figsize=(8, 4))
plt.hist(torch.log1p(degree.float()).numpy(), bins=80)
plt.title("log1p(degree) distribution")
plt.xlabel("log1p(degree)")
plt.ylabel("nodes")
plt.tight_layout()
plt.savefig(OUT_DIR / "degree_distribution.png", dpi=150)
plt.show()

degree_by_class = degree_df.groupby("label")["degree"].agg(["mean", "median", "max", "count"]).reset_index()
display(degree_by_class.sort_values("mean", ascending=False))

## 7. Homophily

Fraction of edges whose endpoints share the same label — a quick sanity check for how much a graph-aware model *should* be able to exploit over a feature-only baseline (MLP).

In [ ]:
same_label = (labels[row] == labels[col]).float()
homophily = float(same_label.mean())
print(f"Edge homophily: {homophily:.4f} over {row.numel():,} directed edges")
(OUT_DIR / "homophily.json").write_text(json.dumps({"edges": int(row.numel()), "homophily": homophily}, indent=2))

## Notes

- Every training script under `graphsage/`, `sagat/`, `gamlp/` defaults to `--split-file split_idx.csv` (the same split loaded above), so results are directly comparable across models.
- See `notebooks/02_compare_results.ipynb` to aggregate `results.json` from `outputs/` across models once you've run some training.